# INFY Custom Data Source (DB-backed)
This notebook loads INFY daily bars from Postgres via a custom QuantConnect `PythonData` source.

In [1]:
# QuantBook Analysis Tool
qb = QuantBook()

# Force reload so notebook always uses latest db_tick_custom_data.py on disk
import importlib
import db_tick_custom_data
importlib.reload(db_tick_custom_data)
print("db_tick_custom_data loaded from:", db_tick_custom_data.__file__)

DbDailyByTradingsymbol = db_tick_custom_data.DbDailyByTradingsymbol

original_log = DbDailyByTradingsymbol._log.__func__
@classmethod
def quiet_cache_hit_log(cls, message: str) -> None:
    if str(message).startswith("Using existing cache for "):
        return
    return original_log(cls, message)

DbDailyByTradingsymbol._log = quiet_cache_hit_log

db_tick_custom_data loaded from: /Lean/Launcher/bin/Debug/Notebooks/db_tick_custom_data.py


In [18]:
# Request INFY data through custom data subscription
qb.set_start_date(2026, 4, 22)
infy_custom = qb.add_data(DbDailyByTradingsymbol, "INFY", Resolution.DAILY)

# Get 360 daily bars, similar to Cell 2 style
history = qb.history(infy_custom.symbol, 360, Resolution.DAILY)
history.tail()

[DbDailyByTradingsymbol] Refreshing cache for INFY
[DbDailyByTradingsymbol] Resolving instrument for tradingsymbol=INFY
[DbDailyByTradingsymbol] Resolved instrument_token=408065
[DbDailyByTradingsymbol] Fetched 5747 rows from daily_candles, date range [2003-01-01 -> 2026-04-24]
[DbDailyByTradingsymbol] Latest row close/volume = 1154.8/41354744 on 2026-04-24
[DbDailyByTradingsymbol] Wrote cache file: /tmp/lean-db-daily/INFY.csv


close    high     low    open  \
symbol                      time                                         
INFY.DbDailyByTradingsymbol 2026-04-16  1319.2  1331.0  1309.0  1322.1   
                            2026-04-17  1318.7  1328.3  1306.1  1313.0   
                            2026-04-20  1312.6  1324.9  1308.1  1320.0   
                            2026-04-21  1313.2  1325.0  1299.3  1310.0   
                            2026-04-22  1257.9  1297.7  1257.0  1295.0   

                                         value      volume  
symbol                      time                            
INFY.DbDailyByTradingsymbol 2026-04-16  1319.2  18527029.0  
                            2026-04-17  1318.7  12064048.0  
                            2026-04-20  1312.6   7172300.0  
                            2026-04-21  1313.2  10057440.0  
                            2026-04-22  1257.9  10635696.0

## Universe from sector-tree + index-constituents

This section builds a custom universe by intersecting:
- Sector subtree for `IT Services` (from `sector-tree`)
- Constituents of `BSE 1000` (from `index-constituents`)

The output universe is a list of NSE codes.

In [ ]:
# Initialize universe from db_universe_custom_data and simulate 3-year selector behavior

import importlib
from datetime import datetime, timedelta

import pandas as pd

import db_universe_custom_data
import utils

importlib.reload(db_universe_custom_data)
importlib.reload(utils)

# === Load real universe payload from DB ===
print("Loading universe payload from DB...")
session_local = db_universe_custom_data.build_session_local()
payload = db_universe_custom_data.build_universe_payload(session_local)
universe_list = payload.get("universe", [])

if not universe_list:
    raise RuntimeError("DB universe payload is empty.")

print(f"Loaded {len(universe_list)} symbols from DB")




Loading universe payload from DB...
Loaded 58 symbols from DB


In [ ]:
# Add top-10 Market Cap universe to QuantBook and include financial data

print(f"\nEvaluating {len(universe_list)} symbols from DB universe...")

# Extract NSE codes from universe
symbols = [row["nse_code"] for row in universe_list]

# Build a quick lookup for financial payload by symbol
universe_lookup = {row["nse_code"]: row for row in universe_list}

# Set date range for analysis
qb.set_start_date(2024, 4, 1)
qb.set_end_date(2026, 4, 24)

# Build ranking by Market Cap from live derived_fundamentals only
ranked_candidates = []
for nse_code in symbols:
    payload_row = universe_lookup.get(nse_code, {})
    financial_data = payload_row.get("processed_financial_data", {})
    market_cap = utils.extract_latest_metric(
        financial_data,
        utils.FINANCIAL_SCORE_SECTION,
        utils.FINANCIAL_SCORE_ROW,
    )

    ranked_candidates.append(
        {
            "nse_code": nse_code,
            "payload_row": payload_row,
            "financial_data": financial_data,
            "market_cap": market_cap,
        }
    )

# Sort descending by Market Cap
ranked_candidates.sort(
    key=lambda x: (x["market_cap"] is not None, x["market_cap"] or 0),
    reverse=True,
)

valid_caps = [x for x in ranked_candidates if x["market_cap"] is not None]
print(f"Symbols with valid Market Cap in live payload: {len(valid_caps)}")

# Keep top 10 by Market Cap where available; fill remaining slots deterministically.
top_10 = valid_caps[:10]
if len(top_10) < 10:
    missing = 10 - len(top_10)
    print(f"Warning: only {len(top_10)} symbols have Market Cap; filling remaining {missing} slots without fallback data.")
    remaining = [x for x in ranked_candidates if x not in top_10]
    top_10.extend(remaining[:missing])

print(f"Selected top {len(top_10)} symbols")
print("Top picks:", [x["nse_code"] for x in top_10])

# Add selected symbols as custom data subscriptions
subscribed_symbols = []
failed = 0

for item in top_10:
    nse_code = item["nse_code"]
    payload_row = item["payload_row"]
    financial_data = item["financial_data"]
    market_cap = item["market_cap"]

    try:
        custom_data = qb.add_data(DbDailyByTradingsymbol, nse_code, Resolution.DAILY)
        subscribed_symbols.append(
            {
                "nse_code": nse_code,
                "symbol": custom_data.symbol,
                "indexes": payload_row.get("indexes", []),
                "sectors": payload_row.get("sectors", []),
                "financial_data": financial_data,
                "financial_sections": sorted(financial_data.keys()),
                "market_cap": market_cap,
                "type": str(custom_data.symbol).split()[1] if len(str(custom_data.symbol).split()) > 1 else "CUSTOM",
            }
        )
    except Exception:
        failed += 1

print(f"\nSuccessfully subscribed to {len(subscribed_symbols)} symbols")
if failed > 0:
    print(f"Failed to subscribe to {failed} symbols")

# Create summary DataFrame
subscriptions_df = pd.DataFrame(subscribed_symbols)
print(f"\nSubscription Summary:")
print(f"  Total symbols: {len(subscriptions_df)}")
print(f"  Symbols with financial data: {int(subscriptions_df['financial_data'].map(bool).sum())}")
print(f"  Unique security types: {subscriptions_df['type'].nunique()}")
print(f"  Symbols with Market Cap: {int(subscriptions_df['market_cap'].notna().sum())}")

print("\nTop 10 subscriptions with financial metadata:")
preview_cols = ["nse_code", "symbol", "financial_sections", "market_cap"]
display(subscriptions_df[preview_cols].sort_values("market_cap", ascending=False).head(10))

# Store for later use
universe_symbols = subscriptions_df["symbol"].tolist()
universe_financial_map = {
    row["nse_code"]: row["financial_data"] for row in subscribed_symbols
}

print(f"\nUniverse ready with {len(universe_symbols)} symbols")


Evaluating 58 symbols from DB universe...


RuntimeError: Only 0 symbols have valid Market Cap in live payload; cannot build top 10 without fallback.

In [21]:
# Obtain price data for universe securities using history and print it

if not universe_symbols:
    raise RuntimeError("Universe symbols are empty. Run Cell 6 first.")

# Rebind custom data loader to Postgres session used by universe builder.
db_tick_custom_data.SessionLocal = db_universe_custom_data.build_session_local()
DbDailyByTradingsymbol._prepared_symbols = set()

lookback_bars = 60
max_symbols = 20  # keep output manageable
symbols_to_query = universe_symbols[:max_symbols]

print(f"Fetching {lookback_bars} daily bars for {len(symbols_to_query)} securities...")

price_frames = []
for symbol in symbols_to_query:
    try:
        raw_hist = qb.history(symbol, lookback_bars, Resolution.DAILY)
        if raw_hist is None or len(raw_hist) == 0:
            continue

        # Normalize history shape across Series/DataFrame/MultiIndex outputs.
        if isinstance(raw_hist, pd.Series):
            hist = raw_hist.to_frame(name="close")
        else:
            hist = raw_hist.copy()

        if isinstance(hist.index, pd.MultiIndex):
            hist = hist.reset_index()
        else:
            hist = hist.reset_index().rename(columns={"index": "time"})

        symbol_code = str(symbol).split(".")[0]
        if "symbol" not in hist.columns:
            hist["symbol"] = symbol_code
        else:
            hist["symbol"] = hist["symbol"].astype(str).str.split(".").str[0]

        # Ensure expected price columns exist for display.
        for col in ["open", "high", "low", "close", "volume", "value"]:
            if col not in hist.columns:
                hist[col] = None

        keep_cols = ["time", "symbol", "open", "high", "low", "close", "volume", "value"]
        price_frames.append(hist[keep_cols])
    except Exception:
        continue

if not price_frames:
    print("No price history retrieved.")
else:
    universe_price_history = pd.concat(price_frames, ignore_index=True)
    universe_price_history = universe_price_history.sort_values(["symbol", "time"])

    print(f"Retrieved price history for {universe_price_history['symbol'].nunique()} securities")
    print(f"Total rows: {len(universe_price_history)}")

    rows_per_symbol = universe_price_history.groupby("symbol").size().sort_values(ascending=False)
    print("\nRows per symbol (top 10):")
    display(rows_per_symbol.head(10).to_frame("rows"))

    print("\nSample price data (first 30 rows):")
    display(universe_price_history.tail(30))

Fetching 60 daily bars for 20 securities...
[DbDailyByTradingsymbol] Refreshing cache for ADANIENSOL
[DbDailyByTradingsymbol] Resolving instrument for tradingsymbol=ADANIENSOL
[DbDailyByTradingsymbol] Resolved instrument_token=2615553
[DbDailyByTradingsymbol] Fetched 2654 rows from daily_candles, date range [2015-07-31 -> 2026-04-24]
[DbDailyByTradingsymbol] Latest row close/volume = 1416.5/7916619 on 2026-04-24
[DbDailyByTradingsymbol] Wrote cache file: /tmp/lean-db-daily/ADANIENSOL.csv
[DbDailyByTradingsymbol] Refreshing cache for ADANIPORTS
[DbDailyByTradingsymbol] Resolving instrument for tradingsymbol=ADANIPORTS
[DbDailyByTradingsymbol] Resolved instrument_token=3861249
[DbDailyByTradingsymbol] Fetched 4544 rows from daily_candles, date range [2007-11-27 -> 2026-04-24]
[DbDailyByTradingsymbol] Latest row close/volume = 1590.0/2282130 on 2026-04-24
[DbDailyByTradingsymbol] Wrote cache file: /tmp/lean-db-daily/ADANIPORTS.csv
[DbDailyByTradingsymbol] Refreshing cache for APOLLOHOSP
[

,rows
symbol,
ADANIENSOL,38
BEL,38
ETERNAL,38
EICHERMOT,38
DRREDDY,38
COALINDIA,38
CIPLA,38
BHARTIARTL,38
BANKINDIA,38



Sample price data (first 30 rows):


,time,symbol,open,high,low,close,volume,value
616,2026-03-06,GRASIM,2701.5,2759.4,2687.2,2718.4,1457401.0,2718.4
617,2026-03-09,GRASIM,2659.0,2692.8,2631.0,2681.2,1019315.0,2681.2
618,2026-03-10,GRASIM,2701.0,2753.0,2694.2,2743.9,1035360.0,2743.9
619,2026-03-11,GRASIM,2726.6,2762.8,2714.8,2735.6,1492588.0,2735.6
620,2026-03-12,GRASIM,2711.2,2727.4,2660.6,2673.1,952036.0,2673.1
621,2026-03-13,GRASIM,2655.0,2664.0,2563.1,2568.6,1309585.0,2568.6
622,2026-03-16,GRASIM,2570.0,2661.4,2565.0,2654.4,1531881.0,2654.4
623,2026-03-17,GRASIM,2655.1,2699.0,2650.0,2683.3,603520.0,2683.3
624,2026-03-18,GRASIM,2699.0,2732.9,2686.7,2723.1,933801.0,2723.1
625,2026-03-19,GRASIM,2700.0,2700.0,2600.1,2607.9,1270396.0,2607.9
